# ShiftLog-Gym Eval + Publish (Notebook 3)

This notebook consumes Notebook 2 outputs, creates final result tables/plots, and optionally uploads artifacts to Hugging Face.

In [ ]:
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip -q install -e . pandas matplotlib seaborn huggingface_hub

In [ ]:
import json
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from huggingface_hub import HfApi, create_repo, login, upload_folder

sns.set_theme(style="whitegrid")

OBS_ROOT = Path("observatory")
RUNS_DIR = OBS_ROOT / "training_runs"
BASELINES_PATH = OBS_ROOT / "baselines.json"
OUTPUT_DIR = Path("artifacts/eval_publish")
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO", "Chirag0096/shiftlog-gym-qwen2.5-3b-memory-policy")
PUBLISH_TO_HF = False

hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    hf_token = getpass("Enter HF_TOKEN (blank to skip upload auth): ").strip()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("Hugging Face login successful.")
else:
    print("HF login skipped.")

In [ ]:
required = {
    "stageA": RUNS_DIR / "training_curves_stageA.csv",
    "stageB": RUNS_DIR / "training_curves_stageB.csv",
    "stageC": RUNS_DIR / "training_curves_stageC.csv",
}

missing = [name for name, path in required.items() if not path.exists()]
if missing:
    print("Missing training curves:", missing)
    print("Run notebook 2 first to generate these files.")
else:
    print("All stage curve CSV files found.")

stage_frames = {}
for name, path in required.items():
    if path.exists():
        stage_frames[name] = pd.read_csv(path)

for name, df in stage_frames.items():
    print(f"\n{name} tail:")
    display(df.tail())

In [ ]:
def plot_stage(df, stage_name):
    fig, ax = plt.subplots(figsize=(10, 4))
    for metric in ["reward_total", "reward_recall", "reward_success", "reward_memory_write", "recall_before_action_rate"]:
        if metric in df.columns:
            ax.plot(df["step"], df[metric], label=metric)
    ax.set_title(f"{stage_name.upper()} Training Curves")
    ax.set_xlabel("step")
    ax.set_ylabel("score")
    ax.legend(loc="best")
    fig.tight_layout()
    out = PLOTS_DIR / f"{stage_name}_curves.png"
    fig.savefig(out, dpi=180)
    plt.show()
    return out

plot_paths = []
for stage_name, df in stage_frames.items():
    plot_paths.append(plot_stage(df, stage_name))
print("Saved plots:")
for p in plot_paths:
    print(p)

In [ ]:
summary_rows = []
for stage_name, df in stage_frames.items():
    if df.empty:
        continue
    summary_rows.append({
        "stage": stage_name,
        "last_reward_total": float(df["reward_total"].iloc[-1]),
        "last_reward_recall": float(df["reward_recall"].iloc[-1]),
        "last_reward_success": float(df["reward_success"].iloc[-1]),
        "last_recall_before_action_rate": float(df["recall_before_action_rate"].iloc[-1]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)
display(summary_df)

In [ ]:
if BASELINES_PATH.exists():
    baselines = json.loads(BASELINES_PATH.read_text(encoding="utf-8"))
else:
    baselines = {"random": {}, "scripted": {}, "llm_base": {}, "trained_llm": {}}

comparison = pd.DataFrame([
    {"agent": "Random Agent", **baselines.get("random", {})},
    {"agent": "Scripted Agent", **baselines.get("scripted", {})},
    {"agent": "Untrained LLM", **baselines.get("llm_base", {})},
    {"agent": "Trained LLM", **baselines.get("trained_llm", {})},
])
comparison.to_csv(OUTPUT_DIR / "before_after_comparison.csv", index=False)
display(comparison)

if "recall_before_action_rate" in comparison.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=comparison, x="agent", y="recall_before_action_rate", ax=ax)
    ax.set_title("Recall Before Action: Baseline vs Trained")
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "recall_before_action_comparison.png", dpi=180)
    plt.show()

In [ ]:
model_card = """
# ShiftLog-Gym Eval Artifacts

This repo contains training evidence for ShiftLog-Gym:
- training curve CSVs for stageA, stageB, stageC
- summary metrics and baseline comparison tables
- plots used in hackathon README/demo
""".strip()
(OUTPUT_DIR / "README.md").write_text(model_card, encoding="utf-8")

print("Prepared output folder:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print("-", p)

## Optional Publish

In [ ]:
if PUBLISH_TO_HF:
    if not os.environ.get("HF_TOKEN"):
        raise ValueError("HF_TOKEN is required to publish. Re-run token cell.")
    create_repo(HF_MODEL_REPO, exist_ok=True, repo_type="model")
    upload_folder(repo_id=HF_MODEL_REPO, repo_type="model", folder_path=str(OUTPUT_DIR))
    print("Published eval artifacts to:", HF_MODEL_REPO)
else:
    print("Publishing skipped. Set PUBLISH_TO_HF=True when ready.")